# Experiment 15 — Native-resolution ablation (RAD-DINO vs DINOv3, NIH-only)

The primary pipeline operates on a common $1024\times1024$ input. This
experiment re-runs exp02 (geometric S4) and exp06 (reticular-fine-$p$=32)
with each model at its native input resolution: RAD-DINO at 518, DINOv3 at
224 and 518. A persistent dissociation at native resolution rules out the
confound that $1024 \to$ native resize (with positional-embedding
interpolation) drives the observed suppression.


In [ ]:
import os, warnings
from pathlib import Path
import pandas as pd

ROOT = Path(os.environ.get('V4_WORK_DIR',
    '/home/saptpurk/embeddings-noise-eliminators/v4_work'))
out_dir = ROOT / 'v4_exp15_native_resolution'
out_dir.mkdir(parents=True, exist_ok=True)

import sys; sys.path.insert(0, str(Path('..').resolve()))
try:
    from common.native_resolution import run_native_resolution_sweep
except Exception as e:
    warnings.warn(f'native_resolution import failed: {e}')
    run_native_resolution_sweep = None

if run_native_resolution_sweep is None:
    pd.DataFrame({'status': ['module_absent']}).to_parquet(
        out_dir / 'exp15_manifest.parquet', index=False)
else:
    rows = run_native_resolution_sweep(
        models=('raddino', 'dinov3'),
        resolutions_by_model={'raddino': (518,), 'dinov3': (224, 518)},
        artifacts=('geometric_S4', 'reticular_fine_p32'),
        dataset='nih', out_dir=out_dir)
    print(f'native-resolution sweep produced {len(rows)} rows')
